# Processing the data (PyTorch)

This notebook walks through **how to prepare raw text data for fine-tuning a BERT model**.
Fine-tuning = taking a pre-trained model and training it a little more on your specific task.

### What we'll cover step by step:
1. A quick end-to-end training loop (to show the goal)
2. Loading a real dataset (MRPC — Microsoft Research Paraphrase Corpus)
3. Tokenizing the dataset (converting text → numbers the model can read)
4. Dynamic padding with a DataCollator (making batches the right shape)

**Key idea:** Models don't read words. They read numbers (token IDs).
Everything in this notebook is about converting raw text into those numbers correctly.

Install the Transformers, Datasets, and Evaluate libraries to run this notebook.

In [1]:
# Install the three core HuggingFace libraries:
#   transformers  — pre-trained models (BERT, GPT-2, T5, etc.) and tokenizers
#   datasets      — fast dataset loading and preprocessing
#   evaluate      — metrics like accuracy, F1, BLEU
# [sentencepiece] installs the SentencePiece tokenizer used by T5, XLNet, etc.
!pip install datasets evaluate transformers[sentencepiece]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.5 MB/s eta 0:00:00


## Step 1 — A quick end-to-end training loop

Before diving into data processing, here's the full picture:
what does a single training step actually look like?
This cell shows the complete loop on just 2 sentences so you can see the goal.

In [2]:
import torch
from torch.optim import AdamW  # AdamW = Adam optimizer with weight decay (better for transformers)
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# ── 1. Load tokenizer + model ────────────────────────────────────────────────
# 'checkpoint' is the name of a pre-trained model on HuggingFace Hub.
# 'bert-base-uncased' = BERT trained on lowercase English text, 110M parameters.
# AutoTokenizer automatically loads the correct tokenizer for this model.
# AutoModelForSequenceClassification loads BERT with a classification head on top
# (an extra linear layer that outputs one score per class).
checkpoint = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)
model = AutoModelForSequenceClassification.from_pretrained(checkpoint)

# ── 2. Tokenize two raw sentences ───────────────────────────────────────────
sequences = [
    "I've been waiting for a HuggingFace course my whole life.",
    "This course is amazing!",
]
# tokenizer() converts text → token IDs + attention masks.
# padding=True  → pad shorter sequences to match the longest one in the batch.
# truncation=True → cut sequences longer than the model's max length (512 for BERT).
# return_tensors='pt' → return PyTorch tensors instead of plain Python lists.
batch = tokenizer(sequences, padding=True, truncation=True, return_tensors="pt")

# ── 3. Add labels ───────────────────────────────────────────────────────────
# The model needs to know the correct answer to compute the loss.
# Here both sentences are labeled 1 ("positive" / "equivalent" — task-dependent).
# torch.tensor([1, 1]) creates a tensor with one label per sentence in the batch.
batch["labels"] = torch.tensor([1, 1])

# ── 4. Set up optimizer ──────────────────────────────────────────────────────
# AdamW will update the model's weights to reduce the loss.
# model.parameters() gives AdamW access to all 110M weights in BERT.
optimizer = AdamW(model.parameters())

# ── 5. Forward pass ─────────────────────────────────────────────────────────
# model(**batch) unpacks the dict as keyword arguments:
#   model(input_ids=..., attention_mask=..., labels=...)
# Because labels are provided, the model automatically computes the loss
# (cross-entropy between predictions and labels).
loss = model(**batch).loss

# ── 6. Backward pass (compute gradients) ─────────────────────────────────────
# loss.backward() calculates how much each weight contributed to the error.
# These gradients tell the optimizer which direction to adjust each weight.
loss.backward()

# ── 7. Update weights ───────────────────────────────────────────────────────
# optimizer.step() uses the gradients to nudge each weight slightly.
# After thousands of these steps on real data, the model learns the task.
optimizer.step()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


## Step 2 — Load a real dataset (MRPC)

Now we replace the two hard-coded sentences with a real dataset.

**MRPC** (Microsoft Research Paraphrase Corpus) is a binary classification task:
given two sentences, predict whether they mean the same thing (paraphrase) or not.
- Label `1` = equivalent (paraphrase)
- Label `0` = not equivalent

It's part of **GLUE** — a benchmark suite of 9 NLP tasks used to evaluate models.

In [3]:
from datasets import load_dataset

# load_dataset("glue", "mrpc") downloads the MRPC subset of the GLUE benchmark.
# The result is a DatasetDict — a dictionary with three splits:
#   raw_datasets['train']      → 3,668 sentence pairs for training
#   raw_datasets['validation'] → 408 pairs for tuning hyperparameters
#   raw_datasets['test']       → 1,725 pairs for final evaluation
raw_datasets = load_dataset("glue", "mrpc")
raw_datasets  # Prints a summary of the dataset structure

README.md: 0.00B [00:00, ?B/s]

mrpc/train-00000-of-00001.parquet:   0%|          | 0.00/649k [00:00<?, ?B/s]

mrpc/validation-00000-of-00001.parquet:   0%|          | 0.00/75.7k [00:00<?, ?B/s]

mrpc/test-00000-of-00001.parquet:   0%|          | 0.00/308k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/3668 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/408 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1725 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 3668
    })
    validation: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 408
    })
    test: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 1725
    })
})

### Inspect a single training example
Let's look at what one data point actually contains.

In [4]:
# Access the training split like a dictionary key.
raw_train_dataset = raw_datasets["train"]

# Index with [0] to get the very first example.
# Each example is a dict with:
#   'sentence1' → first sentence (string)
#   'sentence2' → second sentence (string)
#   'label'     → 0 (not equivalent) or 1 (equivalent)
#   'idx'       → the original row index in the dataset
raw_train_dataset[0]

{'sentence1': 'Amrozi accused his brother , whom he called " the witness " , of deliberately distorting his evidence .',
 'sentence2': 'Referring to him as only " the witness " , Amrozi accused his brother of deliberately distorting his evidence .',
 'label': 1,
 'idx': 0}

### Check column types (features)
`.features` is like a schema — it tells you the data type of each column.

In [5]:
# .features shows the data type of each column in the dataset.
# You'll see:
#   sentence1, sentence2 → Value('string')  — plain text
#   label               → ClassLabel(names=['not_equivalent', 'equivalent'])
#                         This means label=0 → 'not_equivalent', label=1 → 'equivalent'
#   idx                 → Value('int32') — just a row number, not used in training
raw_train_dataset.features

{'sentence1': Value('string'),
 'sentence2': Value('string'),
 'label': ClassLabel(names=['not_equivalent', 'equivalent']),
 'idx': Value('int32')}

## Step 3 — Tokenizing the dataset

The model cannot read raw text — everything must be converted to numbers first.
Tokenization does three things:
1. **Splits** text into tokens (words or subwords)
2. **Maps** each token to an integer ID from the model's vocabulary
3. **Adds special tokens** like `[CLS]` (start) and `[SEP]` (separator/end)

This first version uses `padding='max_length'` — padding every sequence to 512 tokens.
It works but wastes memory. We'll improve this later with dynamic padding.

In [6]:
from transformers import AutoTokenizer

checkpoint = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

# Define a function that tokenizes ONE example (or a batch of examples).
# We pass BOTH sentences together in a single tokenizer call.
# BERT handles two sentences natively using token_type_ids:
#   tokens from sentence1 get type_id=0
#   tokens from sentence2 get type_id=1
# This lets BERT distinguish which sentence each token belongs to.
def tokenize_function(example):
    return tokenizer(
        example["sentence1"],   # First sentence
        example["sentence2"],   # Second sentence (paired input)
        truncation=True,        # Cut if longer than 512 tokens (BERT's max)
        padding="max_length"    # Pad every sequence to exactly 512 tokens
                                # (wasteful but simple — improved later)
    )

# .map() applies tokenize_function to the entire dataset efficiently.
# batched=True means it processes many examples at once (much faster than one-by-one).
# The output is a new dataset with extra columns: input_ids, attention_mask, token_type_ids.
tokenized_datasets = raw_datasets.map(tokenize_function, batched=True)

Map:   0%|          | 0/3668 [00:00<?, ? examples/s]

Map:   0%|          | 0/408 [00:00<?, ? examples/s]

Map:   0%|          | 0/1725 [00:00<?, ? examples/s]

### What does the tokenizer actually produce?
Let's tokenize two raw sentences and inspect every field in the output.

In [7]:
# Tokenize two sentences together to see the raw output.
# The tokenizer returns a dict with three keys:
#
#   input_ids       → list of integer token IDs
#                     e.g. 101=>[CLS], 102=>[SEP], 2023=>'this', etc.
#
#   token_type_ids  → 0 for tokens from sentence1, 1 for sentence2
#                     tells BERT which segment each token belongs to
#
#   attention_mask  → 1 for real tokens, 0 for [PAD] tokens
#                     tells the model to ignore padding positions
inputs = tokenizer("This is the first sentence.", "This is the second one.")
inputs

{'input_ids': [101, 2023, 2003, 1996, 2034, 6251, 1012, 102, 2023, 2003, 1996, 2117, 2028, 1012, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}

### Decode token IDs back to human-readable tokens
This reveals the special tokens BERT adds automatically.

In [8]:
# convert_ids_to_tokens() maps each integer ID back to its string token.
# This is the reverse of tokenization — useful for debugging.
#
# You'll see the BERT special tokens inserted automatically:
#   [CLS]  → always the very first token. BERT uses its hidden state
#             as the sentence-level representation for classification.
#   [SEP]  → separator token. Appears after sentence1 AND after sentence2.
#             Signals the boundary between segments.
#
# Full sequence structure:
#   [CLS] <sentence1 tokens> [SEP] <sentence2 tokens> [SEP]
tokenizer.convert_ids_to_tokens(inputs["input_ids"])

['[CLS]',
 'this',
 'is',
 'the',
 'first',
 'sentence',
 '.',
 '[SEP]',
 'this',
 'is',
 'the',
 'second',
 'one',
 '.',
 '[SEP]']

## Step 4 — Better tokenization: remove fixed-length padding

Padding every sequence to 512 tokens is very wasteful.
Most MRPC sentences are 30–70 tokens long — padding to 512 wastes ~85% of compute!

The better approach: **don't pad during tokenization**.
Instead, pad only at batch time — each batch gets padded to its own longest sequence.
This is called **dynamic padding** and is handled by `DataCollatorWithPadding` below.

In [9]:
# Improved tokenize_function — same as before but WITHOUT padding='max_length'.
# Each tokenized sequence keeps its natural length.
# Sequences in the same batch will be padded to the same length later
# by the DataCollator — only as much as needed, not always 512.
def tokenize_function(example):
    return tokenizer(
        example["sentence1"],
        example["sentence2"],
        truncation=True   # Still truncate anything longer than 512 tokens
        # No padding here! Padding happens dynamically per batch instead.
    )

# Apply to the full dataset. batched=True processes ~1000 examples at a time
# which is much faster than example-by-example (parallelised under the hood).
tokenized_datasets = raw_datasets.map(tokenize_function, batched=True)

Map:   0%|          | 0/3668 [00:00<?, ? examples/s]

Map:   0%|          | 0/408 [00:00<?, ? examples/s]

Map:   0%|          | 0/1725 [00:00<?, ? examples/s]

### Compact version of tokenize_function
Same logic as above — just written on one line. This is the version used going forward.

In [10]:
# This is the final, clean version of the tokenizer function.
# It's identical to the cell above — just condensed to one line.
# truncation=True ensures sequences never exceed BERT's 512-token limit.
def tokenize_function(example):
    return tokenizer(example["sentence1"], example["sentence2"], truncation=True)

### Apply tokenization and inspect the result

In [11]:
# Apply the compact tokenize_function to all three splits (train, validation, test).
# batched=True is important for speed — processes rows in chunks of ~1000.
tokenized_datasets = raw_datasets.map(tokenize_function, batched=True)

# Printing tokenized_datasets shows the updated schema.
# You'll see the original columns (sentence1, sentence2, label, idx)
# PLUS the new columns added by the tokenizer:
#   input_ids       → the token ID sequences (variable length — no padding yet!)
#   token_type_ids  → segment IDs (0=sentence1, 1=sentence2)
#   attention_mask  → all 1s for now (no padding tokens yet)
tokenized_datasets

Map:   0%|          | 0/3668 [00:00<?, ? examples/s]

Map:   0%|          | 0/408 [00:00<?, ? examples/s]

Map:   0%|          | 0/1725 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 3668
    })
    validation: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 408
    })
    test: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 1725
    })
})

## Step 5 — Dynamic padding with DataCollatorWithPadding

A **DataCollator** is a function that takes a list of individual examples
and combines them into a single batch — including padding them to the same length.

`DataCollatorWithPadding` pads sequences **dynamically**:
each batch is padded only to the length of its longest sequence,
not to a fixed 512. This saves significant memory and compute.

In [12]:
from transformers import DataCollatorWithPadding

# DataCollatorWithPadding needs the tokenizer so it knows:
#   1. Which token ID to use for padding (the [PAD] token, ID=0 for BERT)
#   2. Which side to pad on (right-padding by default for BERT)
#   3. How to update the attention_mask (pad positions get mask=0)
#
# This collator will be passed to the DataLoader later.
# Every time a batch is assembled, it pads to that batch's max length.
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

### Verify that sequences have different lengths before padding
This confirms dynamic padding is needed — the sequences are NOT all the same length.

In [13]:
# Grab the first 8 examples from the training set.
# tokenized_datasets["train"][:8] returns a dict of lists (one list per column).
samples = tokenized_datasets["train"][:8]

# Remove columns the model doesn't need:
#   'idx'       → just a row number, not a model input
#   'sentence1' → raw text, already converted to input_ids
#   'sentence2' → same — we don't need the strings anymore
# We keep: input_ids, attention_mask, token_type_ids, label
samples = {k: v for k, v in samples.items() if k not in ["idx", "sentence1", "sentence2"]}

# Print the length of each of the 8 input_ids sequences.
# You'll see different numbers (e.g. [50, 59, 47, 67, 59, 50, 62, 32]).
# This proves the sequences are different lengths — padding is required
# to stack them into a rectangular batch tensor.
[len(x) for x in samples["input_ids"]]

[50, 59, 47, 67, 59, 50, 62, 32]

### Apply the DataCollator and check the resulting batch shape
After collating, all sequences in the batch have the same length — padded to the longest one.

In [14]:
# data_collator(samples) takes our list of variable-length sequences
# and pads them all to the length of the LONGEST sequence in this batch.
# It returns a dict of PyTorch tensors — one per column.
batch = data_collator(samples)

# Print the shape of each tensor in the batch.
# Shape format: (batch_size, sequence_length)
#   batch_size      = 8 (we took 8 examples)
#   sequence_length = max length in this batch (e.g. 67 from the list above)
#
# All tensors have the SAME shape — this is required for GPU matrix operations.
# Shorter sequences have been padded with [PAD] tokens (ID=0);
# their attention_mask entries are set to 0 so the model ignores them.
{k: v.shape for k, v in batch.items()}

{'input_ids': torch.Size([8, 67]),
 'token_type_ids': torch.Size([8, 67]),
 'attention_mask': torch.Size([8, 67]),
 'labels': torch.Size([8])}